# 1. Data Loading - PubMedQA Dataset

This notebook downloads the PubMedQA subset from the [AQ-MedAI/RAG-QA-Leaderboard](https://huggingface.co/datasets/AQ-MedAI/RAG-QA-Leaderboard) dataset on HuggingFace, extracts the relevant documents from the large document pool, and saves processed data locally for use in subsequent RAG pipeline notebooks.

**Dataset overview:**
- `pubmed.jsonl` - 500 queries with fields: `id`, `query`, `golden_doc`, `reference`, `ground_truth`
- `documents_pool.json` - ~740 MB pool of documents mapped as `Dict[str, str]`
- `golden_doc`: list of relevant document IDs (0-6 per row)
- `reference`: list of 50 distractor document IDs per row
- `ground_truth`: "yes", "no", or "maybe"

In [1]:
import json
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
import pandas as pd

In [2]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

## 1.1 Download PubMedQA subset

In [3]:
# Download pubmed.jsonl from HuggingFace
pubmed_path = hf_hub_download(
    repo_id="AQ-MedAI/RAG-QA-Leaderboard",
    filename="final_data/pubmed.jsonl",
    repo_type="dataset"
)
print(f"Downloaded pubmed.jsonl to: {pubmed_path}")

# Read line by line
queries = []
with open(pubmed_path, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            queries.append(json.loads(line))

print(f"Loaded {len(queries)} queries")

pubmed.jsonl: 0.00B [00:00, ?B/s]

Downloaded pubmed.jsonl to: /Users/devsquad/.cache/huggingface/hub/datasets--AQ-MedAI--RAG-QA-Leaderboard/snapshots/5788a13a8a551f5c4f7ea92f901fe2e924dccb69/final_data/pubmed.jsonl
Loaded 500 queries


In [4]:
# Explore the data
print(f"Number of queries: {len(queries)}")
print("\n--- First row ---")
print(json.dumps(queries[0], indent=2))

# Distribution of ground_truth values
print("\n--- ground_truth distribution ---")
gt_dist = pd.Series([q["ground_truth"] for q in queries]).value_counts()
print(gt_dist)
print(f"\nTotal: {gt_dist.sum()}")

Number of queries: 500

--- First row ---
{
  "id": "pubmed_1",
  "query": "Is anorectal endosonography valuable in dyschesia?",
  "golden_doc": [
    "Document_47183",
    "Document_21825"
  ],
  "reference": [
    "Document_627762",
    "Document_216720",
    "Document_1072783",
    "Document_618203",
    "Document_310665",
    "Document_927896",
    "Document_481451",
    "Document_320893",
    "Document_87253",
    "Document_833705",
    "Document_489828",
    "Document_532423",
    "Document_129761",
    "Document_280861",
    "Document_903477",
    "Document_1004568",
    "Document_830237",
    "Document_1001355",
    "Document_174766",
    "Document_161560",
    "Document_428917",
    "Document_339184",
    "Document_344779",
    "Document_641810",
    "Document_201157",
    "Document_866298",
    "Document_613972",
    "Document_149272",
    "Document_848977",
    "Document_618414",
    "Document_500519",
    "Document_657506",
    "Document_912905",
    "Document_128257",
    

## 1.2 Collect unique document IDs

In [5]:
# Collect ALL unique doc IDs from both golden_doc and reference fields
golden_doc_ids = set()
reference_doc_ids = set()

for q in queries:
    for doc_id in q["golden_doc"]:
        golden_doc_ids.add(doc_id)
    for doc_id in q["reference"]:
        reference_doc_ids.add(doc_id)

all_doc_ids = golden_doc_ids | reference_doc_ids

print(f"Unique doc IDs from golden_doc: {len(golden_doc_ids):,}")
print(f"Unique doc IDs from reference:  {len(reference_doc_ids):,}")
print(f"Total unique doc IDs:           {len(all_doc_ids):,}")

Unique doc IDs from golden_doc: 860
Unique doc IDs from reference:  21,043
Total unique doc IDs:           21,903


## 1.3 Download and extract documents from pool

In [6]:
print("Downloading documents_pool.json (740MB, may take a few minutes)...")
pool_path = hf_hub_download(
    repo_id="AQ-MedAI/RAG-QA-Leaderboard",
    filename="final_data/documents_pool.json",
    repo_type="dataset"
)

print("Loading document pool...")
with open(pool_path, "r") as f:
    full_pool = json.load(f)

print(f"Total documents in pool: {len(full_pool):,}")

# Extract only the documents we need
documents = {doc_id: full_pool[doc_id] for doc_id in all_doc_ids if doc_id in full_pool}
print(f"Extracted {len(documents):,} documents for PubMedQA subset")

# Check for missing docs
missing = all_doc_ids - set(documents.keys())
if missing:
    print(f"WARNING: {len(missing)} document IDs not found in pool: {missing}")
else:
    print("All document IDs found in pool!")

# Free memory
del full_pool

final_data/documents_pool.json:   0%|          | 0.00/740M [00:00<?, ?B/s]

Loading document pool...


Total documents in pool: 1,089,245
Extracted 21,903 documents for PubMedQA subset
All document IDs found in pool!


In [7]:
# Quick stats about document lengths
doc_lengths = [len(text) for text in documents.values()]

print(f"Document length statistics (in characters):")
print(f"  Min:  {min(doc_lengths):,}")
print(f"  Max:  {max(doc_lengths):,}")
print(f"  Mean: {sum(doc_lengths) / len(doc_lengths):,.0f}")

# Preview one document
sample_id = list(documents.keys())[0]
print(f"\n--- Preview of {sample_id} (first 200 chars) ---")
print(documents[sample_id][:200])

Document length statistics (in characters):
  Min:  63
  Max:  1,470
  Mean: 693

--- Preview of Document_1053429 (first 200 chars) ---
title: Sham peer review
content: peer review could draw out the process by legal maneuvering, and the fairness of a peer review that has been unduly delayed has been called into question. Many medical


## 1.4 Save processed data

In [8]:
# Save documents
docs_path = DATA_DIR / "pubmed_documents.json"
with open(docs_path, "w") as f:
    json.dump(documents, f)
print(f"Saved {len(documents):,} documents to {docs_path}")

# Save queries
queries_path = DATA_DIR / "pubmedqa.jsonl"
with open(queries_path, "w") as f:
    for q in queries:
        f.write(json.dumps(q) + "\n")
print(f"Saved {len(queries)} queries to {queries_path}")

Saved 21,903 documents to ../data/pubmed_documents.json
Saved 500 queries to ../data/pubmedqa.jsonl


## 1.5 Verification

In [9]:
# Verify all golden docs are in our extracted documents
golden_coverage = 0
total_golden = 0
for q in queries:
    for gd in q["golden_doc"]:
        total_golden += 1
        if gd in documents:
            golden_coverage += 1

print(f"Golden doc coverage: {golden_coverage}/{total_golden} ({golden_coverage/total_golden*100:.1f}%)")

queries_with_golden = sum(1 for q in queries if len(q["golden_doc"]) > 0)
print(f"Queries with at least one golden doc: {queries_with_golden}/{len(queries)}")
print(f"\nFiles saved:")
print(f"  - {docs_path} ({docs_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"  - {queries_path} ({queries_path.stat().st_size / 1024:.1f} KB)")

Golden doc coverage: 860/860 (100.0%)
Queries with at least one golden doc: 488/500

Files saved:
  - ../data/pubmed_documents.json (15.0 MB)
  - ../data/pubmedqa.jsonl (569.1 KB)


## Summary

In this notebook we:

1. **Downloaded** the `pubmed.jsonl` file (500 queries) from the AQ-MedAI/RAG-QA-Leaderboard dataset on HuggingFace.
2. **Collected** all unique document IDs referenced by both `golden_doc` and `reference` fields.
3. **Downloaded** the full `documents_pool.json` (740 MB) and **extracted** only the documents needed for the PubMedQA subset.
4. **Saved** the processed data locally:
   - `data/pubmed_documents.json` - extracted documents
   - `data/pubmedqa.jsonl` - query data
5. **Verified** that all golden document IDs are present in the extracted document set.

**Next notebook:** Chunking and embedding the documents for vector search.